<a href="https://colab.research.google.com/github/Abdullah-Sayeed7/projects/blob/main/dbcnn_SAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
img1 = cv2.imread("/content/san_1.bmp", cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread("/content/san_2.bmp", cv2.IMREAD_GRAYSCALE)
gt   = cv2.imread("/content/san_gt.bmp", cv2.IMREAD_GRAYSCALE)

img1_f = img1.astype(np.float32) + 1e-10
img2_f = img2.astype(np.float32) + 1e-10

def normalize(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-10)

log_ratio = normalize(np.abs(np.log(img1_f) - np.log(img2_f)))
img1_n    = normalize(img1_f)
img2_n    = normalize(img2_f)
labels    = (gt > 128).astype(np.int64).flatten()

print(f"Changed: {labels.sum()} | Unchanged: {(labels==0).sum()}")

Changed: 4685 | Unchanged: 60851


In [ ]:
def extract_patches_fast(image, patch_size=7):
    h, w  = image.shape
    pad   = patch_size // 2
    padded = np.pad(image, pad, mode='reflect').astype(np.float32)

    shape   = (h, w, patch_size, patch_size)
    strides = (padded.strides[0], padded.strides[1],
               padded.strides[0], padded.strides[1])

    patches = np.lib.stride_tricks.as_strided(padded, shape=shape, strides=strides)
    return patches.reshape(-1, patch_size, patch_size)

PATCH_SIZE = 7
patches = extract_patches_fast(log_ratio, PATCH_SIZE)
patches.shape

(65536, 7, 7)

In [ ]:
H, W = 256, 256

label_grid = labels.reshape(H, W)
patch_grid = patches.reshape(H, W, PATCH_SIZE, PATCH_SIZE)

split_col = int(W * 0.7)

train_mask = np.zeros((H, W), dtype=bool)
train_mask[:, :split_col] = True
test_mask = ~train_mask

X_tr = torch.tensor(patch_grid[train_mask][:, np.newaxis], dtype=torch.float32)
X_te = torch.tensor(patch_grid[test_mask][:, np.newaxis],  dtype=torch.float32)
y_tr = torch.tensor(label_grid[train_mask], dtype=torch.long)
y_te = torch.tensor(label_grid[test_mask],  dtype=torch.long)

X = torch.tensor(patches[:, np.newaxis], dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.long)

print(f"Train: {X_tr.shape[0]} pixels (columns 0-{split_col-1})")
print(f"Test:  {X_te.shape[0]} pixels (columns {split_col}-{W-1})")
print(f"Train changed: {y_tr.sum()} | Test changed: {y_te.sum()}")

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=512, shuffle=True)

Train: 45824 pixels (columns 0-178)
Test:  19712 pixels (columns 179-255)
Train changed: 4133 | Test changed: 552


In [ ]:
P_img1 = extract_patches_fast(img1_n, PATCH_SIZE)
P_img2 = extract_patches_fast(img2_n, PATCH_SIZE)
print(P_img1.shape, P_img2.shape)

(65536, 7, 7) (65536, 7, 7)


In [ ]:
P1_grid = P_img1.reshape(H, W, PATCH_SIZE, PATCH_SIZE)
P2_grid = P_img2.reshape(H, W, PATCH_SIZE, PATCH_SIZE)

S1_tr = torch.tensor(P1_grid[train_mask][:, np.newaxis], dtype=torch.float32)
S2_tr = torch.tensor(P2_grid[train_mask][:, np.newaxis], dtype=torch.float32)
S1_te = torch.tensor(P1_grid[test_mask][:, np.newaxis],  dtype=torch.float32)
S2_te = torch.tensor(P2_grid[test_mask][:, np.newaxis],  dtype=torch.float32)

S1_all = torch.tensor(P_img1[:, np.newaxis], dtype=torch.float32)
S2_all = torch.tensor(P_img2[:, np.newaxis], dtype=torch.float32)

from torch.utils.data import Dataset
class DualDS(Dataset):
    def __init__(self, x1, x2, y): self.x1, self.x2, self.y = x1, x2, y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x1[i], self.x2[i], self.y[i]

dual_loader = DataLoader(DualDS(S1_tr, S2_tr, y_tr), batch_size=256, shuffle=True)

In [ ]:
class DualBranchCNN(nn.Module):
    def __init__(self):
        super().__init__()
        def make_branch():
            return nn.Sequential(
                nn.Conv2d(1, 30, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(30, 60, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(60, 120, 3, padding=1), nn.ReLU(),
            )
        self.branch1 = make_branch()
        self.branch2 = make_branch()
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(240, 240), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(240, 2)
        )
    def forward(self, x1, x2):
        f1 = self.branch1(x1)
        f2 = self.branch2(x2)
        merged = torch.cat([f1, f2], dim=1)
        return self.classifier(merged)

model = DualBranchCNN().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 221,282


In [ ]:
def evaluate_dual(model, X1, X2, y_true, label=""):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X1), 1024):
            out = model(X1[i:i+1024].to(device), X2[i:i+1024].to(device))
            preds.extend(torch.argmax(out, 1).cpu().numpy())
    preds = np.array(preds)
    true  = y_true.numpy()

    TP=((preds==1)&(true==1)).sum(); TN=((preds==0)&(true==0)).sum()
    FP=((preds==1)&(true==0)).sum(); FN=((preds==0)&(true==1)).sum()
    OE=FP+FN; PCC=(TP+TN)/len(true)*100
    d=(TP+FP)*(FP+TN)+(TP+FN)*(FN+TN)
    KC=2*(TP*TN-FP*FN)/d*100 if d else 0

    print(f"\n {label} \nFP:{FP} FN:{FN} OE:{OE}\nPCC:{PCC:.2f}% KC:{KC:.2f}%")
    return preds

evaluate_dual(model, S1_te, S2_te, y_te, "TEST SET")
all_preds = evaluate_dual(model, S1_all, S2_all, y, "FULL IMAGE")


── TEST SET ──
FP:130 FN:2 OE:132
PCC:99.33% KC:88.94%

── FULL IMAGE ──
FP:883 FN:16 OE:899
PCC:98.63% KC:90.48%


In [ ]:
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 10.0]).to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

EPOCHS = 30
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x1b, x2b, yb in dual_loader:
        x1b, x2b, yb = x1b.to(device), x2b.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x1b, x2b), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg = total_loss / len(dual_loader)
    scheduler.step(avg)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg:.4f}")

Epoch   5/30 | Loss: 0.0442
Epoch  10/30 | Loss: 0.0409
Epoch  15/30 | Loss: 0.0377
Epoch  20/30 | Loss: 0.0310
Epoch  25/30 | Loss: 0.0315
Epoch  30/30 | Loss: 0.0287


In [1]:
# Reducing weight to 5

In [ ]:
model = DualBranchCNN().to(device)

criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 5.0]).to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

EPOCHS = 30
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x1b, x2b, yb in dual_loader:
        x1b, x2b, yb = x1b.to(device), x2b.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x1b, x2b), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg = total_loss / len(dual_loader)
    scheduler.step(avg)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg:.4f}")

evaluate_dual(model, S1_te, S2_te, y_te, "TEST SET")
evaluate_dual(model, S1_all, S2_all, y, "FULL IMAGE")

Epoch   5/30 | Loss: 0.0421
Epoch  10/30 | Loss: 0.0373
Epoch  15/30 | Loss: 0.0349
Epoch  20/30 | Loss: 0.0349
Epoch  25/30 | Loss: 0.0285
Epoch  30/30 | Loss: 0.0283

── TEST SET ──
FP:77 FN:12 OE:89
PCC:99.55% KC:92.15%

── FULL IMAGE ──
FP:499 FN:41 OE:540
PCC:99.18% KC:94.06%


array([0, 0, 0, ..., 0, 0, 0])